# 🚀 Đánh Giá Năng Lực Tuân Thủ Ràng Buộc (Instruction Following) Trên Tiny LLM: Qwen2.5-0.5B-Instruct

Notebook này thực hiện **đánh giá toàn diện (Inference & Verification)** mô hình siêu nhỏ **`Qwen/Qwen2.5-0.5B-Instruct`** trên các tập dữ liệu:
* **Task 1 (Hard-only):** 29 quy tắc hình thức Seen + 24 quy tắc OOD Unseen.
* **Task 2 (Hybrid):** Kết hợp Hard Constraints (cấu trúc) + Soft Constraints (ngữ nghĩa/văn phong).
* **CRPL Variants:** Các biến thể ranh giới (*Containment*, *Partial Overlap*, *Disjoint*) để đo lường độ nhạy ranh giới và kiểm tra hiện tượng học vẹt.

---
### 🛠️ Kiến trúc Hệ thống Giám khảo (Two-Tier Verifier):
1. **Student Policy:** `Qwen/Qwen2.5-0.5B-Instruct` (0.49 tỷ tham số, chạy siêu nhanh ~0.5s/câu trên GPU T4 miễn phí của Colab hoặc CPU).
2. **Hard Verifier (Python Rule Engine):** Chấm 100% tự động theo chuẩn IFEval & IFBench (độ dài, từ khóa, dấu câu, format, bảng biểu, ngôn ngữ...).
3. **Soft Verifier (LLM-as-a-Judge):** Gemini API đóng vai trò thẩm định viên ngữ nghĩa (Semantic Completeness & Professional Tone).
4. **Hard-Priority Multiplicative Gating:** Nếu vi phạm hoàn toàn Hard Rules ($R_{\text{hard}} = 0$) hoặc lạc đề ($R_{\text{soft}} < 0.5$) $\rightarrow$ Reward bị Gate về $0.0$ (chống Reward Hacking).


In [ ]:
# @title 1. Cài đặt các thư viện cần thiết
!pip install -q transformers accelerate pandas tqdm torch langdetect


In [ ]:
# @title 2. Thiết lập Google API Key (Dành cho Gemini LLM-as-a-Judge)
import os

# Nhập API Key của bạn (miễn phí tại https://aistudio.google.com/):
GOOGLE_API_KEY = "YOUR_GEMINI_API_KEY" # @param {type:"string"}

if GOOGLE_API_KEY == "YOUR_GEMINI_API_KEY" or not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    except Exception:
        GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

if not GOOGLE_API_KEY:
    print("⚠️ CẢNH BÁO: Chưa có Google API Key! Nếu chỉ chấm Hard Rules thì không sao, nhưng nếu chấm Soft Rules thì Gemini sẽ báo lỗi.")
else:
    print("✓ Google API Key đã được thiết lập thành công!")


In [ ]:
# @title 3. Nạp Dataset Đánh Giá (Hỗ trợ Hybrid, CRPL & Hard-only)
import os
import json

# Tùy chọn: Kết nối Google Drive nếu đang chạy trên Google Colab
MOUNT_DRIVE = True # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/vinfast/RLVR"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        if not os.path.isdir('/content/drive/MyDrive'):
            print("Đang kết nối tới Google Drive...")
            drive.mount('/content/drive')
    except ImportError:
        print("Môi trường không phải Colab, bỏ qua mount Drive.")

# Danh sách dataset hỗ trợ đánh giá:
DATASET_FILENAME = "crpl_task2_hybrid_alpaca_gpt4.json" # @param ["crpl_task2_hybrid_alpaca_gpt4.json", "task2_hybrid_alpaca_gpt4.json", "task2_hybrid_gpt4_self_instruct.json", "task2_eval_unseen_constraints.json", "crpl_alpaca_gpt4.json", "augmented_alpaca_gpt4.json"]

# Tìm file ở các đường dẫn khả dĩ:
candidate_paths = [
    DATASET_FILENAME,
    os.path.join(DRIVE_DIR, DATASET_FILENAME),
    os.path.join("/content", DATASET_FILENAME),
    os.path.join("datasets/seen", DATASET_FILENAME),
    os.path.join("datasets/unseen", DATASET_FILENAME),
]

dataset_path = None
for p in candidate_paths:
    if os.path.isfile(p):
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError(f"Không tìm thấy file {DATASET_FILENAME} ở bất kỳ vị trí nào: {candidate_paths}")

print(f"-> Đang nạp dataset từ: {dataset_path}")
dataset = []
if dataset_path.endswith(".jsonl"):
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                dataset.append(json.loads(line.strip()))
else:
    with open(dataset_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        dataset = data if isinstance(data, list) else [data]

print(f"✓ Đã nạp thành công {len(dataset)} mẫu dữ liệu!")
print(f"  • Mẫu đầu tiên có key: {dataset[0].get('key', 'unknown')}")
print(f"  • Có Soft Constraints: {'soft_constraints' in dataset[0]}")
print(f"  • Có CRPL Metadata: {'crpl_metadata' in dataset[0]}")


In [ ]:
# @title 4. Engine Code Verifier (Chấm Toàn Diện Hard Constraints: 29 Seen + 24 Unseen OOD)
import re
import json
import string
from collections import Counter
from langdetect import detect_langs, DetectorFactory
DetectorFactory.seed = 0

def split_sentences(text):
    clean = text.strip()
    parts = re.split(r'(?<!\b\d)(?<=[.!?])\s+', clean)
    return [p.strip() for p in parts if p.strip()]

def verify_hard_constraint_detailed(inst_id, kwargs, response_text):
    """
    Xác thực ràng buộc cứng và trả về tuple: (passed: bool, note: str)
    """
    text = response_text.strip()
    
    # 1. Quotation
    if inst_id == "startend:quotation":
        passed = (text.startswith('"') and text.endswith('"')) or (text.startswith('“') and text.endswith('”'))
        return passed, "Được bao bọc trong ngoặc kép" if passed else "Không có ngoặc kép ở đầu và cuối"
    
    # 2. Number of sentences
    if inst_id == "length_constraints:number_sentences":
        num = kwargs.get("num_sentences", 0)
        rel = kwargs.get("relation", "exactly")
        sentences = split_sentences(text)
        count = len(sentences)
        if rel == "less than": passed = count < num
        elif rel == "at least": passed = count >= num
        else: passed = (count == num)
        return passed, f"Số câu: {count} (yêu cầu {rel} {num})"
        
    # 3. Postscript
    if inst_id == "detectable_content:postscript":
        marker = kwargs.get("postscript_marker", "P.S.")
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        if not lines: return False, "Phản hồi rỗng"
        passed = lines[-1].startswith(marker) or lines[-1].startswith("Tái bút:")
        return passed, "Có tái bút ở dòng cuối" if passed else "Dòng cuối không bắt đầu bằng P.S. hoặc Tái bút:"
        
    # 4. Words count
    if inst_id == "length_constraints:number_words":
        words = len(text.split())
        num = kwargs.get("num_words", 0)
        rel = kwargs.get("relation", "less than")
        if rel == "less than": passed = words < num
        elif rel == "at least": passed = words >= num
        elif rel == "range": passed = kwargs.get("min_words", 0) <= words <= kwargs.get("max_words", 9999)
        else: passed = (words == num)
        return passed, f"Số từ: {words} (yêu cầu {rel} {num})"
        
    # 5. Paragraphs count
    if inst_id == "length_constraints:number_paragraphs":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        req = kwargs.get("num_paragraphs", 0)
        passed = len(paras) == req
        return passed, f"Số đoạn văn: {len(paras)} (yêu cầu {req})"
        
    # 6. Bullet lists
    if inst_id == "detectable_format:number_bullet_lists":
        bullets = re.findall(r'(?m)^\s*[*-]\s+', text)
        req = kwargs.get("num_bullets", 0)
        passed = len(bullets) == req
        return passed, f"Số gạch đầu dòng: {len(bullets)} (yêu cầu {req})"
        
    # 7. Numbered list
    if inst_id == "format:numbered_list":
        items = re.findall(r'(?m)^\s*\d+\.\s+', text)
        req = kwargs.get("num_items", 0)
        passed = len(items) == req
        return passed, f"Số mục đánh số: {len(items)} (yêu cầu {req})"
        
    # 8. JSON Format
    if inst_id == "detectable_format:json_format":
        try:
            clean_j = text.strip().strip('`')
            if clean_j.startswith('json'): clean_j = clean_j[4:].strip()
            json.loads(clean_j)
            return True, "JSON hợp lệ"
        except Exception as e:
            return False, f"JSON không hợp lệ: {str(e)[:50]}"
        
    # 9. Title (<<Title>> or # Title)
    if inst_id == "detectable_format:title":
        has_chevrons = bool(re.search(r'<<[^>]+>>', text))
        has_header = bool(re.search(r'(?m)^#\s+.+', text))
        passed = has_chevrons or has_header
        return passed, "Có tiêu đề định dạng hợp lệ" if passed else "Không tìm thấy tiêu đề dạng <<...>> hoặc # ..."
        
    # 10. Keyword existence
    if inst_id == "keywords:existence":
        kws = kwargs.get("keywords", [])
        lower = text.lower()
        missing = [w for w in kws if not re.search(rf'\b{re.escape(w.lower())}\b', lower)]
        passed = len(missing) == 0
        return passed, "Đủ từ khóa" if passed else f"Thiếu từ khóa: {missing}"
        
    # 11. Keyword frequency
    if inst_id == "keywords:frequency":
        word = kwargs.get("keyword", "").lower()
        freq = kwargs.get("frequency", 0)
        rel = kwargs.get("relation", "exactly")
        actual = len(re.findall(rf'\b{re.escape(word)}\b', text.lower()))
        if rel == "at least": passed = actual >= freq
        elif rel == "at most": passed = actual <= freq
        else: passed = (actual == freq)
        return passed, f"Từ '{word}' xuất hiện {actual} lần (yêu cầu {rel} {freq})"
        
    # 12. Letter frequency
    if inst_id == "keywords:letter_frequency":
        letter = kwargs.get("letter", "").lower()
        freq = kwargs.get("let_frequency", 0)
        rel = kwargs.get("let_relation", "at least")
        actual = text.lower().count(letter)
        if rel == "at least": passed = actual >= freq
        elif rel == "at most": passed = actual <= freq
        else: passed = (actual == freq)
        return passed, f"Chữ '{letter}' xuất hiện {actual} lần (yêu cầu {rel} {freq})"
        
    # 13. Forbidden words
    if inst_id == "keywords:forbidden_words":
        forb = kwargs.get("forbidden_words", [])
        lower = text.lower()
        found = [w for w in forb if re.search(rf'\b{re.escape(w.lower())}\b', lower)]
        passed = len(found) == 0
        return passed, "Không chứa từ cấm" if passed else f"Chứa từ cấm: {found}"
        
    # 14. End checker
    if inst_id == "startend:end_checker":
        phrase = kwargs.get("end_phrase", "").strip()
        passed = text.strip().endswith(phrase) or text.strip().rstrip(".!?").endswith(phrase)
        return passed, "Kết thúc đúng cụm từ" if passed else f"Không kết thúc bằng '{phrase}'"
        
    # 15. Multiple sections
    if inst_id == "detectable_format:multiple_sections":
        n = kwargs.get("num_sections", 0)
        spliter = kwargs.get("section_spliter", "Section")
        sections = re.findall(rf'(?i){re.escape(spliter)}\s*\d+', text)
        passed = len(sections) >= n
        return passed, f"Tìm thấy {len(sections)} phần (yêu cầu ít nhất {n})"
        
    # 16. Highlighted sections
    if inst_id == "detectable_format:number_highlighted_sections":
        n = kwargs.get("num_highlights", 0)
        highlights = re.findall(r'\*{1,2}[^*\n]+\*{1,2}', text)
        passed = len(highlights) >= n
        return passed, f"Tìm thấy {len(highlights)} đoạn highlight (yêu cầu {n})"
        
    # 17. No comma
    if inst_id == "punctuation:no_comma":
        comma_count = text.count(',') + text.count('，')
        passed = comma_count == 0
        return passed, "Không có dấu phẩy" if passed else f"Chứa {comma_count} dấu phẩy"
        
    # 18. XML wrapper
    if inst_id == "format:xml_wrapper":
        passed = text.startswith("<response>") and text.endswith("</response>")
        return passed, "Bao bọc thẻ <response> đúng" if passed else "Thiếu hoặc sai thẻ <response>"
        
    # 19. All caps / Lower
    if inst_id == "change_case:english_capital":
        passed = text.isupper()
        return passed, "Toàn bộ là chữ hoa" if passed else "Có chứa chữ thường"
    if inst_id == "change_case:english_lowercase":
        passed = text.islower()
        return passed, "Toàn bộ là chữ thường" if passed else "Có chứa chữ hoa"

    # 20. Capital words frequency
    if inst_id == "change_case:capital_word_frequency":
        caps = re.findall(r'\b[A-Z]{2,}\b', text)
        rel = kwargs.get("relation", "at least")
        tgt = kwargs.get("capital_words", 2)
        if rel == "at least": passed = len(caps) >= tgt
        elif rel == "at most": passed = len(caps) <= tgt
        elif rel == "less than": passed = len(caps) < tgt
        else: passed = (len(caps) == tgt)
        return passed, f"Có {len(caps)} từ ALL CAPS (yêu cầu {rel} {tgt})"

    # 21. Response language (Chuẩn hóa langdetect, hỗ trợ đa chế độ CRPL)
    if inst_id == "language:response_language":
        LANG_MAP = {"Vietnamese": "vi", "French": "fr", "English": "en", "Spanish": "es", "German": "de"}
        def _lang_ratio(t, code):
            try:
                return next((l.prob for l in detect_langs(t) if l.lang == code), 0.0)
            except Exception:
                return 0.0

        mode = kwargs.get("mode", "all")
        lang_name = kwargs.get("language", "Vietnamese")
        code = LANG_MAP.get(lang_name, "vi")

        if mode == "all":
            prob = _lang_ratio(text, code)
            passed = prob >= 0.80
            return passed, f"Độ khớp ngôn ngữ {lang_name} ({code}): {prob:.1%}"

        if mode == "any_of":
            vi = _lang_ratio(text, "vi")
            en = _lang_ratio(text, "en")
            passed = vi >= 0.80 or (vi >= 0.20 and en >= 0.20)
            return passed, f"Tỷ lệ ngôn ngữ: VN={vi:.1%}, EN={en:.1%}"

        if mode == "first_half_second_half":
            paras = [p for p in re.split(r'\n\s*\n', text) if p.strip()]
            if len(paras) < 2:
                paras = split_sentences(text)
                if len(paras) < 2: return False, "Văn bản quá ngắn để chia nửa ngôn ngữ"
            mid = len(paras) // 2
            first, second = "\n".join(paras[:mid]), "\n".join(paras[mid:])
            r1 = _lang_ratio(first, "vi")
            r2 = _lang_ratio(second, "en")
            passed = r1 >= 0.65 and r2 >= 0.65
            return passed, f"Nửa đầu (VN): {r1:.1%}, Nửa sau (EN): {r2:.1%}"
        return False, f"Chế độ mode '{mode}' không hợp lệ"

    # 22. Paragraph nth first word
    if inst_id == "length_constraints:nth_paragraph_first_word":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        nth = kwargs.get("nth_paragraph", 1) - 1
        expected = kwargs.get("first_word", "").lower()
        if nth < len(paras):
            w = re.findall(r'\b\w+\b', paras[nth].lower())
            actual = w[0] if w else ""
            passed = (actual == expected)
            return passed, f"Đoạn {nth+1} bắt đầu bằng '{actual}' (yêu cầu '{expected}')"
        return False, f"Không đủ {nth+1} đoạn văn"

    # 23. Number placeholders (Khắc phục n == 0 và hỗ trợ Unicode tiếng Việt)
    if inst_id == "detectable_content:number_placeholders":
        PLACEHOLDER_RE = re.compile(r'\[[^\[\]\n]+\]')
        n = kwargs.get("num_placeholders", 1)
        found = PLACEHOLDER_RE.findall(text)
        if n == 0:
            passed = ('[' not in text and ']' not in text)
            return passed, "Không có ngoặc vuông" if passed else f"Tìm thấy ngoặc vuông: {found}"
        passed = len(found) >= n
        return passed, f"Tìm thấy {len(found)} placeholders (yêu cầu ít nhất {n})"

    # 24. Constrained response
    if inst_id == "detectable_format:constrained_response":
        opts = kwargs.get("constrained_responses", [])
        lower = text.lower()
        matched = [o for o in opts if re.search(rf'\b{re.escape(o.lower())}\b', lower)]
        passed = len(matched) > 0
        return passed, f"Khớp lựa chọn: {matched}" if passed else f"Không chứa lựa chọn nào trong: {opts}"

    # 25. Two responses
    if inst_id == "combination:two_responses":
        passed = "******" in text
        return passed, "Có dấu phân cách ******" if passed else "Thiếu dấu phân cách ******"

    # 26. Repeat prompt
    if inst_id == "combination:repeat_prompt":
        prompt_to_repeat = kwargs.get("prompt_to_repeat", "").strip()
        if prompt_to_repeat:
            first_para = text.split('\n')[0].strip()
            passed = prompt_to_repeat.lower() in first_para.lower()
            return passed, "Đã lặp lại prompt ở dòng đầu" if passed else "Không tìm thấy prompt ở đầu bài"
        passed = len(text.split('\n')) >= 2
        return passed, "Đã lặp lại câu hỏi" if passed else "Không đủ số dòng lặp"

    # 27. Table format
    if inst_id == "detectable_format:table_format":
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        table_lines = [l for l in lines if l.startswith('|') and l.endswith('|')]
        passed = len(table_lines) >= 2
        return passed, f"Tìm thấy {len(table_lines)} dòng bảng Markdown (yêu cầu >= 2)"

    # 28. No period
    if inst_id == "punctuation:no_period":
        period_count = text.count('.') + text.count('。')
        passed = period_count == 0
        return passed, "Không có dấu chấm" if passed else f"Chứa {period_count} dấu chấm"

    # ================== UNSEEN POOL (24 Curated IFBench OOD) ==================
    if inst_id == "count:unique_word_count":
        words = re.findall(r'\b\w+\b', text.lower())
        u_count = len(set(words))
        req = kwargs.get("num_unique_words", 0)
        return u_count >= req, f"Số từ độc nhất: {u_count} (yêu cầu {req})"

    if inst_id == "format:parentheses":
        nested = re.findall(r'\(\([^\(\)]+\)\)', text)
        req = kwargs.get("num_nested", 1)
        return len(nested) >= req, f"Số ngoặc đôi (()): {len(nested)} (yêu cầu {req})"

    if inst_id == "words:palindrome":
        palindromes = kwargs.get("palindromes", [])
        lower = text.lower()
        found = [p for p in palindromes if re.search(rf'\b{re.escape(p.lower())}\b', lower)]
        return len(found) == len(palindromes), f"Từ đối xứng tìm thấy: {found}/{palindromes}"

    if inst_id == "sentence:increment":
        sentences = split_sentences(text)
        if len(sentences) < 2: return False, "Cần ít nhất 2 câu"
        lens = [len(s.split()) for s in sentences]
        passed = all(lens[i] < lens[i+1] for i in range(len(lens)-1))
        return passed, f"Độ dài từng câu: {lens}"

    if inst_id == "words:start_verb":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        if not paras: return False, "Phản hồi rỗng"
        COMMON_VERBS = {'eat', 'exercise', 'get', 'drink', 'sleep', 'maintain', 'prioritize', 'practice', 'focus', 'ensure', 'avoid', 'include', 'engage', 'build', 'create', 'develop', 'run', 'walk', 'read', 'write', 'start', 'begin', 'take', 'make', 'do', 'use', 'stay', 'keep', 'choose', 'aim', 'limit', 'reduce', 'boost', 'protect', 'adopt', 'schedule', 'cultivate', 'foster', 'incorporate', 'strive'}
        for para in paras:
            first_word = re.findall(r'\b\w+\b', para.lower())
            if not first_word or (first_word[0] not in COMMON_VERBS and not (first_word[0].endswith('ing') or first_word[0].endswith('ed'))):
                return False, f"Đoạn văn không bắt đầu bằng động từ hợp lệ: '{first_word[0] if first_word else 'Rỗng'}'"
        return True, "Mọi đoạn đều bắt đầu bằng động từ"

    if inst_id == "count:punctuation":
        found = set(c for c in text if c in string.punctuation)
        req = kwargs.get("num_punctuation_types", 4)
        return len(found) >= req, f"Số loại dấu câu: {len(found)} (yêu cầu {req})"

    if inst_id == "words:alphabet_loop":
        words = re.findall(r'\b[a-zA-Z]', text.lower())
        if len(words) < 3: return False, "Không đủ số từ"
        passed = all((ord(words[i]) - ord('a')) == (ord(words[i-1]) - ord('a') + 1) % 26 for i in range(1, len(words)))
        return passed, "Vòng lặp bảng chữ cái" if passed else "Thứ tự bảng chữ cái không liên tục"

    if inst_id == "words:prime_lengths":
        primes = {2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31}
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        if not words: return False, "Không có từ"
        passed = all(len(w) in primes for w in words)
        return passed, "Mọi từ đều có độ dài là số nguyên tố" if passed else "Có từ có độ dài không phải số nguyên tố"

    if inst_id == "format:nested_quotes":
        max_d, stack = 0, []
        for ch in text:
            if ch in ('"', "'"):
                if stack and stack[-1] == ch: stack.pop()
                else: stack.append(ch); max_d = max(max_d, len(stack))
        return max_d >= 3, f"Độ sâu ngoặc kép lồng nhau: {max_d} (yêu cầu >= 3)"

    if inst_id == "count:numbers_count":
        nums = re.findall(r'\b\d+\b', text)
        req = kwargs.get("num_numbers", 3)
        return len(nums) == req, f"Số lượng số tìm thấy: {len(nums)} (yêu cầu {req})"

    if inst_id == "format:title_case":
        words = re.findall(r'\b[a-zA-Z]+\b', text)
        if not words: return False, "Không có từ"
        cap_count = sum(1 for w in words if w[0].isupper())
        ratio = cap_count / len(words)
        return ratio >= 0.75, f"Tỷ lệ viết hoa đầu từ: {ratio:.1%} (yêu cầu >= 75%)"

    if inst_id == "sentence:last_word_first_next":
        sentences = split_sentences(text)
        if len(sentences) < 2: return False, "Cần ít nhất 2 câu"
        for i in range(len(sentences) - 1):
            w_last = re.findall(r'\b\w+\b', sentences[i].lower())
            w_first = re.findall(r'\b\w+\b', sentences[i+1].lower())
            if not w_last or not w_first or w_last[-1] != w_first[0]:
                return False, f"Câu {i+1} kết thúc bằng '{w_last[-1] if w_last else ''}' nhưng câu {i+2} bắt đầu bằng '{w_first[0] if w_first else ''}'"
        return True, "Từ nối giữa các câu hợp lệ"

    if inst_id == "words:no_consecutive_first_letter":
        words = re.findall(r'\b[a-zA-Z]', text.lower())
        if len(words) < 2: return True, "Đạt"
        passed = all(words[i] != words[i+1] for i in range(len(words) - 1))
        return passed, "Không có 2 từ liên tiếp cùng chữ cái đầu" if passed else "Có từ liên tiếp trùng chữ cái đầu"

    if inst_id == "words:limited_repeat":
        words = re.findall(r'\b\w+\b', text.lower())
        if not words: return False, "Rỗng"
        max_rep = Counter(words).most_common(1)[0][1]
        req = kwargs.get("max_repeats", 4)
        return max_rep <= req, f"Từ lặp nhiều nhất: {max_rep} lần (cho phép tối đa {req})"

    if inst_id == "sentence:word_count_step":
        sentences = split_sentences(text)
        if len(sentences) < 2: return False, "Cần ít nhất 2 câu"
        lens = [len(re.findall(r'\b\w+\b', s)) for s in sentences]
        step = kwargs.get("step", 2)
        passed = all(lens[i+1] - lens[i] == step for i in range(len(lens)-1))
        return passed, f"Độ dài các câu: {lens} (bước nhảy yêu cầu {step})"

    if inst_id == "format:date_format_list":
        dates = re.findall(r'\b\d{4}-\d{2}-\d{2}\b', text)
        req = kwargs.get("min_dates", 2)
        return len(dates) >= req, f"Tìm thấy {len(dates)} ngày chuẩn YYYY-MM-DD (yêu cầu >= {req})"

    if inst_id == "format:csv_format":
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        req = kwargs.get("num_rows", 3)
        if len(lines) < req: return False, f"Chỉ có {len(lines)} dòng (yêu cầu {req})"
        passed = all(',' in l for l in lines)
        return passed, "Định dạng CSV hợp lệ" if passed else "Có dòng thiếu dấu phẩy phân tách"

    if inst_id == "format:output_template":
        passed = ("My Answer:" in text and "My Conclusion:" in text and "Future Outlook:" in text)
        return passed, "Đủ 3 đề mục template" if passed else "Thiếu một trong 3 đề mục yêu cầu"

    if inst_id == "words:paragraph_last_first_match":
        paras = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
        if not paras: return False, "Rỗng"
        for p in paras:
            words = re.findall(r'\b\w+\b', p.lower())
            if len(words) < 2 or words[0] != words[-1]:
                return False, f"Đoạn có từ đầu '{words[0] if words else ''}' khác từ cuối '{words[-1] if words else ''}'"
        return True, "Từ đầu và cuối các đoạn đều trùng khớp"

    if inst_id == "words:conjunction_count":
        conj_list = {'and', 'but', 'for', 'nor', 'or', 'so', 'yet'}
        words = set(re.findall(r'\b\w+\b', text.lower()))
        used = words.intersection(conj_list)
        req = kwargs.get("min_conjunctions", 3)
        return len(used) >= req, f"Số liên từ: {len(used)} ({used}) (yêu cầu {req})"

    if inst_id == "format:indent_stairs":
        lines = [l for l in text.split('\n') if l.strip()]
        if len(lines) < 3: return False, "Cần ít nhất 3 dòng"
        indents = [len(l) - len(l.lstrip(' ')) for l in lines]
        passed = all(indents[i] < indents[i+1] for i in range(len(indents)-1))
        return passed, f"Độ thụt lề: {indents}"

    if inst_id == "format:special_bullet":
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        bullet_lines = [l for l in lines if l.startswith('->')]
        return len(bullet_lines) >= 2, f"Số dòng bắt đầu bằng '->': {len(bullet_lines)} (yêu cầu >= 2)"

    if inst_id == "manipulation:no_whitespace":
        has_ws = any(c in string.whitespace for c in text)
        return not has_ws, "Không có khoảng trắng" if not has_ws else "Chứa khoảng trắng"

    if inst_id == "sentence:symbol_end":
        sentences = split_sentences(text)
        if not sentences: return False, "Rỗng"
        passed = all(s.endswith('!*') or '!*' in s[-5:] for s in sentences)
        return passed, "Mọi câu đều kết thúc bằng !*" if passed else "Có câu không kết thúc bằng !*"

    raise ValueError(f"Unknown constraint id: {inst_id}")

def verify_hard_constraint(inst_id, kwargs, response_text):
    passed, _ = verify_hard_constraint_detailed(inst_id, kwargs, response_text)
    return passed

print("✓ Engine Code Verifier (Phiên bản v2 - Hỗ trợ Chi Tiết Từng Ràng Buộc) đã sẵn sàng!")


In [ ]:
# @title 5. Engine Soft Verifier (Gemini LLM-as-a-Judge)
import urllib.request
import json
import time

def evaluate_soft_constraint_gemini(response_text, soft_constraint, api_key, model="gemini-2.5-flash"):
    """
    Đánh giá tiêu chí mềm và trả về: (score: Optional[int], reason: str, error: Optional[str])
    """
    if not api_key or api_key == "YOUR_GEMINI_API_KEY":
        return None, "Chưa thiết lập Google API Key", "MISSING_API_KEY"
    
    if isinstance(soft_constraint, str):
        try:
            soft_constraint = json.loads(soft_constraint)
        except Exception:
            soft_constraint = {"description": soft_constraint, "eval_rubric": soft_constraint}
            
    if not isinstance(soft_constraint, dict):
        return None, "Định dạng soft constraint không hợp lệ", "INVALID_FORMAT"
        
    c_desc = soft_constraint.get("description", "")
    rubric = soft_constraint.get("eval_rubric", "")
    
    judge_prompt = f"""You are an impartial, expert evaluation judge assessing an LLM's response quality for instruction following.

CRITERION TO EVALUATE:
{c_desc}

EVALUATION RUBRIC / SCORING STANDARD:
{rubric}

CANDIDATE RESPONSE TO ASSESS:
"""
{response_text}
"""

DECISION TASK:
Determine strictly whether the response satisfies the rubric criteria above.
Return ONLY a valid JSON object matching this exact schema:
{{
  "score": 1 or 0,
  "reasoning": "<1-2 concise sentences explaining the rationale>"
}}
"""
    payload = {
        "contents": [{"parts": [{"text": judge_prompt}]}],
        "generationConfig": {
            "temperature": 0.0,
            "responseMimeType": "application/json"
        }
    }
    data = json.dumps(payload).encode("utf-8")
    
    fallback_models = [model, "gemini-2.0-flash", "gemini-1.5-flash"]
    last_err = None
    
    for current_model in fallback_models:
        current_url = f"https://generativelanguage.googleapis.com/v1beta/models/{current_model}:generateContent?key={api_key}"
        for attempt in range(2):
            try:
                req = urllib.request.Request(current_url, data=data, headers={"Content-Type": "application/json"}, method="POST")
                with urllib.request.urlopen(req, timeout=20) as resp:
                    res = json.loads(resp.read().decode("utf-8"))
                    raw_text = res["candidates"][0]["content"]["parts"][0]["text"]
                    parsed = json.loads(raw_text)
                    score = int(parsed.get("score", 0))
                    reason = parsed.get("reasoning", "")
                    return score, reason, None
            except Exception as e:
                last_err = str(e)
                time.sleep(1.0 * (attempt + 1))
            
    return None, f"Lỗi gọi Judge API: {last_err}", last_err

print("✓ Engine Soft Verifier (Gemini LLM-as-a-Judge) đã sẵn sàng!")


In [ ]:
# @title 6. Tải Tiny LLM & Cấu hình Tham số Sinh (Inference Hyperparameters)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Chọn mô hình siêu nhỏ (Tiny LLM):
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct" # @param ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "meta-llama/Llama-3.2-1B-Instruct"]

# 2. Tham số sinh câu trả lời (Generation Hyperparameters):
NUM_TEST_SAMPLES = 20 # @param {type:"integer"}
MAX_NEW_TOKENS = 512 # @param {type:"integer"}
DO_SAMPLE = False # @param {type:"boolean"}
TEMPERATURE = 0.0 # @param {type:"number"}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang tải mô hình {MODEL_ID} lên thiết bị: {device}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

if device == "cpu":
    model = model.to("cpu")

num_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"✓ Tải mô hình {MODEL_ID} thành công ({num_params:.2f}B tham số).")
print(f"✓ Cấu hình chạy: {NUM_TEST_SAMPLES if NUM_TEST_SAMPLES > 0 else 'Toàn bộ'} mẫu | max_tokens={MAX_NEW_TOKENS} | do_sample={DO_SAMPLE} | temp={TEMPERATURE}")


In [ ]:
# @title 7. Chạy Inference & Chấm điểm (Cấu trúc đầu ra chuẩn hóa)
import time
from tqdm.auto import tqdm

# Chuẩn bị dữ liệu đánh giá:
if "NUM_TEST_SAMPLES" in locals() and NUM_TEST_SAMPLES and NUM_TEST_SAMPLES > 0:
    test_data = dataset[:NUM_TEST_SAMPLES]
else:
    test_data = dataset

eval_results = []
model_name_short = MODEL_ID.split('/')[-1]

print(f"Bắt đầu đánh giá {len(test_data)} mẫu với {MODEL_ID}...")

for idx, item in enumerate(tqdm(test_data, desc="Evaluating")):
    prompt = item["prompt"]
    key = item.get("key", f"sample_{idx}")
    crpl_meta = item.get("crpl_metadata", {})
    base_id = key.rsplit("_crpl_", 1)[0] if "_crpl_" in key else key
    rel = item.get("crpl_relationship", crpl_meta.get("relationship", "none"))
    
    # 1. Sinh câu trả lời từ LLM & Đo lường Token
    messages = [{"role": "user", "content": prompt}]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    input_token_len = model_inputs.input_ids.shape[1]
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE if DO_SAMPLE else None
        )
    
    output_token_ids = generated_ids[0][input_token_len:]
    output_tokens = len(output_token_ids)
    finish_reason = "length" if output_tokens >= MAX_NEW_TOKENS else "stop"
    qwen_response = tokenizer.decode(output_token_ids, skip_special_tokens=True).strip()
    
    # 2. Chấm điểm Hard Constraints (Python Rule Verifier - Structured Items)
    hc = item.get("hard_constraints", item)
    c_ids = hc.get("constraint_ids", item.get("constraint_ids", []))
    gts = hc.get("ground_truth", item.get("ground_truth", []))
    
    if len(gts) == 1 and isinstance(gts[0], dict) and "kwargs" in gts[0]:
        kwargs_list = gts[0]["kwargs"]
    else:
        kwargs_list = gts
        
    hard_items = []
    hard_passed = 0
    for c_id, gt in zip(c_ids, kwargs_list):
        gt_eval = dict(gt) if isinstance(gt, dict) else {}
        if c_id == "language:response_language" and "mode" not in gt_eval:
            if rel == "containment": gt_eval["mode"] = "any_of"
            elif rel == "partial_overlap": gt_eval["mode"] = "first_half_second_half"
            else: gt_eval["mode"] = "all"
            
        passed, note = verify_hard_constraint_detailed(c_id, gt_eval, qwen_response)
        if passed: hard_passed += 1
        hard_items.append({
            "id": c_id,
            "passed": passed,
            "kwargs": gt_eval,
            "note": note
        })
        
    hard_total = len(c_ids)
    r_hard = (hard_passed / hard_total) if hard_total > 0 else 1.0
    
    # 3. Chấm điểm Soft Constraints (Gemini LLM-as-a-Judge - Structured Items)
    soft_data = item.get("soft_constraints", [])
    if isinstance(soft_data, dict):
        soft_list = soft_data.get("constraints", [])
    elif isinstance(soft_data, list):
        soft_list = soft_data
    else:
        soft_list = []
        
    soft_items = []
    judge_error = None
    soft_passed = 0
    
    for sc in soft_list:
        sc_id = sc.get("id", "soft_criterion")
        score, reason, err = evaluate_soft_constraint_gemini(qwen_response, sc, api_key=GOOGLE_API_KEY)
        if err is not None:
            judge_error = err
        if score is not None:
            soft_passed += score
        soft_items.append({
            "id": sc_id,
            "score": score,
            "reason": reason
        })
        time.sleep(0.3)
        
    soft_total = len(soft_items)
    
    # 4. Tính toán Reward (Raw vs Gated) & Xác định Gate Status rõ ràng
    if judge_error is not None:
        soft_status = "error"
        r_soft = None
        r_raw = round(r_hard, 3)
        r_gated = 0.0
        gate_status = "JUDGE_ERROR"
    elif soft_total == 0:
        soft_status = "ok"
        r_soft = 1.0
        r_raw = round(r_hard, 3)
        r_gated = round(r_hard, 2)
        gate_status = "PASSED" if r_hard > 0 else "HARD_FAIL"
    else:
        soft_status = "ok"
        r_soft = round(soft_passed / soft_total, 2)
        r_raw = round(0.75 * r_hard + 0.25 * r_soft, 3)
        if r_hard == 0.0:
            r_gated = 0.0
            gate_status = "HARD_FAIL"
        elif r_soft < 0.5:
            r_gated = 0.0
            gate_status = "SOFT_FAIL"
        else:
            r_gated = round(0.75 * r_hard + 0.25 * r_soft, 2)
            gate_status = "PASSED"
            
    # 5. Đóng gói Bản ghi Đánh giá Theo Schema Cấu Trúc Chuẩn Hóa
    record = {
        "key": key,
        "base_id": base_id,
        "relationship": rel,
        "perturbed_constraint_index": crpl_meta.get("perturbed_constraint_index", 0),
        "base_instruction": item.get("base_instruction", ""),
        "generation": {
            "model": model_name_short,
            "temperature": TEMPERATURE if DO_SAMPLE else 0.0,
            "max_new_tokens": MAX_NEW_TOKENS,
            "finish_reason": finish_reason,
            "output_tokens": output_tokens
        },
        "response": qwen_response,
        "hard": {
            "verifier_version": "v2",
            "score": round(r_hard, 2),
            "passed": hard_passed,
            "total": hard_total,
            "items": hard_items
        },
        "soft": {
            "judge_model": "gemini-2.5-flash",
            "rubric_version": "v1",
            "status": soft_status,
            "score": r_soft,
            "items": soft_items,
            "error": judge_error
        },
        "reward": {
            "raw": r_raw,
            "gated": r_gated,
            "status": gate_status
        }
    }
    eval_results.append(record)

print(f"✓ ĐÃ HOÀN TẤT ĐÁNH GIÁ {len(eval_results)} MẪU!")


In [ ]:
# @title 8. Bảng Tổng Kết Kết Quả & Thống Kê Phân Tích
import pandas as pd
from collections import defaultdict

# 1. Bảng dữ liệu phẳng hóa cho Pandas
flat_rows = []
for r in eval_results:
    flat_rows.append({
        "key": r["key"],
        "base_id": r["base_id"],
        "relationship": r["relationship"],
        "hard_score": r["hard"]["score"],
        "soft_score": r["soft"]["score"],
        "reward_raw": r["reward"]["raw"],
        "reward_gated": r["reward"]["gated"],
        "status": r["reward"]["status"],
        "output_tokens": r["generation"]["output_tokens"],
        "finish_reason": r["generation"]["finish_reason"]
    })

df = pd.DataFrame(flat_rows)

print("============================================================")
print(f"📊 BÁO CÁO TỔNG HỢP ĐÁNH GIÁ TINY LLM: {MODEL_ID}")
print(f"• Tổng số mẫu đánh giá: {len(df)}")
print(f"• Strict Accuracy (Đạt 100% Hard Rules): {(df['hard_score'] == 1.0).mean():.1%}")
print(f"• Điểm Hard Rules trung bình: {df['hard_score'].mean():.2%}")
valid_soft = df['soft_score'].dropna()
soft_mean_str = f'{valid_soft.mean():.2%}' if len(valid_soft) > 0 else 'N/A (Chưa chấm hoặc lỗi key)'
print(f'• Điểm Soft Rules trung bình: {soft_mean_str}')
print(f"• Điểm Reward Raw (Chưa Gate) trung bình: {df['reward_raw'].mean():.3f}")
print(f"• Điểm Reward Gated (Sau Gate) trung bình: {df['reward_gated'].mean():.3f}")
print("• Phân phối trạng thái Reward:")
for stat, cnt in df['status'].value_counts().items():
    print(f"    - {stat:15s}: {cnt}/{len(df)} ({cnt/len(df):.1%})")
print("============================================================\n")

# 2. Thống kê theo quan hệ CRPL (Boundary Robustness)
if "relationship" in df.columns and (df['relationship'] != 'none').any():
    print("--- 🔬 PHÂN TÍCH THEO QUAN HỆ CRPL (BOUNDARY SENSITIVITY) ---")
    crpl_summary = df.groupby('relationship').agg(
        So_Mau=('key', 'count'),
        Strict_Pass=('hard_score', lambda s: f"{(s == 1.0).mean():.1%}"),
        Hard_Score_TB=('hard_score', lambda s: f"{s.mean():.2%}"),
        Reward_Raw_TB=('reward_raw', lambda s: f"{s.mean():.3f}"),
        Reward_Gated_TB=('reward_gated', lambda s: f"{s.mean():.3f}"),
        Pass_Gate=('status', lambda s: f"{(s == 'PASSED').mean():.1%}")
    )
    print(crpl_summary.to_string())
    print("\n")

# 3. Thống kê tỷ lệ đạt theo từng loại Hard Constraint ID cụ thể
c_stats = defaultdict(lambda: {"passed": 0, "total": 0})
for r in eval_results:
    for item in r["hard"]["items"]:
        cid = item["id"]
        c_stats[cid]["total"] += 1
        if item["passed"]:
            c_stats[cid]["passed"] += 1

print("--- 📌 TỶ LỆ ĐẠT THEO TỪNG LOẠI HARD CONSTRAINT ID ---")
c_rows = []
for cid, st in sorted(c_stats.items(), key=lambda x: x[1]["passed"]/x[1]["total"], reverse=True):
    rate = st["passed"] / st["total"]
    c_rows.append({"Constraint ID": cid, "Passed": st["passed"], "Total": st["total"], "Pass Rate": f"{rate:.1%}"})
df_c = pd.DataFrame(c_rows)
print(df_c.to_string(index=False))
print("\n")

# Hiển thị 10 mẫu kết quả đầu tiên
display_cols = ["key", "base_id", "relationship", "hard_score", "soft_score", "reward_raw", "reward_gated", "status"]
display(df[display_cols].head(10))


In [ ]:
# @title 9. Xuất Đầy Đủ 4 Định Dạng (JSONL, JSON, CSV & summary.json)
import os
import json
import pandas as pd
from collections import defaultdict

if os.path.isdir("/content/drive/MyDrive"):
    SAVE_DIR = "/content/drive/MyDrive/vinfast/RLVR"
else:
    SAVE_DIR = "results"

os.makedirs(SAVE_DIR, exist_ok=True)

base_name = DATASET_FILENAME.replace(".jsonl", "").replace(".json", "")
model_short = MODEL_ID.split("/")[-1].lower()

out_jsonl = os.path.join(SAVE_DIR, f"{base_name}_{model_short}_eval_results.jsonl")
out_json = os.path.join(SAVE_DIR, f"{base_name}_{model_short}_eval_results.json")
out_csv = os.path.join(SAVE_DIR, f"{base_name}_{model_short}_eval_results.csv")
out_summary = os.path.join(SAVE_DIR, f"{base_name}_{model_short}_summary.json")

# 1. Xuất file streaming JSONL
with open(out_jsonl, "w", encoding="utf-8") as f:
    for record in eval_results:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# 2. Xuất file JSON format đẹp
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

# 3. Xuất file CSV phẳng
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

# 4. Tạo và xuất file summary.json
c_stats = defaultdict(lambda: {"passed": 0, "total": 0})
for r in eval_results:
    for item in r["hard"]["items"]:
        cid = item["id"]
        c_stats[cid]["total"] += 1
        if item["passed"]:
            c_stats[cid]["passed"] += 1

crpl_metrics = {}
if "relationship" in df.columns:
    for rel_name, group in df.groupby("relationship"):
        crpl_metrics[rel_name] = {
            "samples": len(group),
            "strict_pass_rate": round(float((group["hard_score"] == 1.0).mean()), 3),
            "mean_hard_score": round(float(group["hard_score"].mean()), 3),
            "mean_reward_raw": round(float(group["reward_raw"].mean()), 3),
            "mean_reward_gated": round(float(group["reward_gated"].mean()), 3),
            "passed_rate": round(float((group["status"] == "PASSED").mean()), 3)
        }

valid_soft = df["soft_score"].dropna()
summary_report = {
    "model": MODEL_ID,
    "dataset": DATASET_FILENAME,
    "total_samples": len(eval_results),
    "hyperparameters": {
        "temperature": TEMPERATURE if DO_SAMPLE else 0.0,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE
    },
    "metrics_overall": {
        "strict_accuracy": round(float((df["hard_score"] == 1.0).mean()), 3),
        "mean_hard_score": round(float(df["hard_score"].mean()), 3),
        "mean_soft_score": round(float(valid_soft.mean()), 3) if len(valid_soft) > 0 else None,
        "mean_reward_raw": round(float(df["reward_raw"].mean()), 3),
        "mean_reward_gated": round(float(df["reward_gated"].mean()), 3),
        "status_distribution": df["status"].value_counts().to_dict()
    },
    "metrics_by_crpl_relationship": crpl_metrics,
    "metrics_by_constraint_id": {
        cid: {
            "passed": st["passed"],
            "total": st["total"],
            "pass_rate": round(st["passed"] / st["total"], 3)
        }
        for cid, st in sorted(c_stats.items(), key=lambda x: x[1]["passed"]/x[1]["total"], reverse=True)
    }
}

with open(out_summary, "w", encoding="utf-8") as f:
    json.dump(summary_report, f, ensure_ascii=False, indent=2)

print("✓ ĐÃ XUẤT ĐẦY ĐỦ 4 ĐỊNH DẠNG:")
print(f"  1. JSONL (Streaming): {out_jsonl}")
print(f"  2. JSON (Formatted): {out_json}")
print(f"  3. CSV (Tabular):     {out_csv}")
print(f"  4. Summary JSON:      {out_summary}")
